## Import dependencies

In [150]:
!pip install bs4 requests dotenv pandas -q
!pip install lxml --break-system-packages -q
from bs4 import BeautifulSoup
import requests
import os
from dotenv import load_dotenv
import pandas as pd
load_dotenv()

print("Dependencies Imported")

Dependencies Imported


## Log in to CSES

In [151]:
LOGIN_URL = "https://cses.fi/login"
TARGET_URL = "https://cses.fi/problemset/"
session = requests.Session()

session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
})

In [152]:
login_page = session.get("https://cses.fi/login")
soup = BeautifulSoup(login_page.text, "lxml")

csrf_token = soup.find("input", {"name": "csrf_token"})["value"]

login_data = {
    "csrf_token": csrf_token,
    "nick": os.getenv("CSES_USERNAME"),
    "pass": os.getenv("CSES_PASSWORD"),
}

In [153]:
login_response = session.post(LOGIN_URL, data=login_data)

In [154]:
print(login_response.url)
check = session.get("https://cses.fi/")
soup_check = BeautifulSoup(check.text, "lxml")

if soup_check.find("a", href=lambda h: h and "logout" in h):
    print("Login Successful.")
else:
    raise Exception("Login Unsuccessful. Check your credentials!")

https://cses.fi/
Login Successful.


In [155]:
# Fetch problem set html page
target_response = session.get(TARGET_URL)

In [156]:
soup = BeautifulSoup(target_response.text, 'lxml')

## Parse HTML into a dictionary + dataframe

In [157]:
# Segment name: [(Problem name, Solved by me, total solved, total attempts)]
ans = {}

df = pd.DataFrame({
    'Header': [],
    'Problem Name': [],
    'Solved By Me': [],
    'Solve Count' : [],
    'Attempt Count': [] 
})


for header in headers:
    n = header.text

    ans[n] = []
    task_list = header.next_sibling.select('li')
    tasks = task_list
    for task in tasks:
        done = True if len(task.select('span.full')) >= 1 else False
        task_name = task.select_one('a').text
        
        solve_count_numbers = task.select_one('span.detail').text

        solve_count_numbers = solve_count_numbers.split('/')

        attempts = solve_count_numbers[-1].strip()
        solve_count = solve_count_numbers[0].strip()

        solve_count, attempts = int(solve_count), int(attempts)

        row = (task_name, done, solve_count, attempts)

        ans[n].append(row)

        row = pd.DataFrame({
            'Header': [n],
            'Problem Name': [row[0]],
            'Solved By Me': [row[1]],
            'Solve Count' : [row[2]],
            'Attempt Count': [row[3]]
        })

        df = pd.concat([df, row], ignore_index=True)

df['Header'] = df['Header'].astype(str)
df['Problem Name'] = df['Problem Name'].astype(str)
df['Solved By Me'] = df['Solved By Me'].astype(bool)
df['Solve Count'] = df['Solve Count'].astype(int)
df['Attempt Count'] = df['Attempt Count'].astype(int)

In [158]:
df

,Header,Problem Name,Solved By Me,Solve Count,Attempt Count
0,Introductory Problems,Weird Algorithm,True,170410,178106
1,Introductory Problems,Missing Number,True,147213,153960
2,Introductory Problems,Repetitions,True,127983,132868
3,Introductory Problems,Increasing Array,True,120605,124678
4,Introductory Problems,Permutations,True,105797,108879
...,...,...,...,...,...
395,Additional Problems II,Maximum Building II,False,475,584
396,Additional Problems II,Stick Divisions,False,4035,4712
397,Additional Problems II,Stick Difference,False,82,296
398,Additional Problems II,Coding Company,False,1605,2096


## Analysis

In [159]:
data = df.copy()

In [160]:
solved = data['Solved By Me'].sum()
total = len(data)
print("You have solved: ", solved , "/", total, f"({str(solved/total * 100)}%).")
print()
print("Solve Count By Category: \n")

summary = data.groupby('Header').agg(
    total_solved = ('Solved By Me', 'sum'),
    total = ('Header','count')
).rename(
    columns={
        'total_solved': 'Total Solved',
        'total': 'Total Problems' 
    }
).sort_values('Total Solved', ascending=False)

summary['Percentage'] = (summary['Total Solved'] / summary['Total Problems'] * 100).round(2).astype(str) + " %"


display(summary)

You have solved:  160 / 400 (40.0%).

Solve Count By Category: 



,Total Solved,Total Problems,Percentage
Header,,,
Sorting and Searching,35,35,100.0 %
Graph Algorithms,26,36,72.22 %
Introductory Problems,23,24,95.83 %
Dynamic Programming,22,23,95.65 %
Range Queries,15,25,60.0 %
Mathematics,12,37,32.43 %
Sliding Window Problems,9,11,81.82 %
Tree Algorithms,8,16,50.0 %
String Algorithms,3,21,14.29 %


In [161]:
data['Difficulty'] = data['Solve Count'] / data['Attempt Count']

data['Difficulty'] = data['Difficulty'].apply(lambda x: str(round(x*100 *100) / 100) + " %")

In [162]:
print("Should solve next: \n")

unsolved = data.loc[data['Solved By Me'] == False]

unsolved.sort_values(by=['Solve Count', 'Difficulty'], ascending=False).head(10)

Should solve next: 



,Header,Problem Name,Solved By Me,Solve Count,Attempt Count,Difficulty
146,Tree Algorithms,Tree Distances I,False,22181,23757,93.37 %
147,Tree Algorithms,Tree Distances II,False,18090,18900,95.71 %
169,Mathematics,Binomial Coefficients,False,11797,13035,90.5 %
23,Introductory Problems,Grid Path Description,False,11209,14395,77.87 %
151,Tree Algorithms,Counting Paths,False,10804,11411,94.68 %
155,Tree Algorithms,Distinct Colors,False,9926,10872,91.3 %
217,Geometry,Point Location Test,False,8294,9213,90.02 %
102,Graph Algorithms,Planets Cycles,False,8083,8957,90.24 %
156,Tree Algorithms,Finding a Centroid,False,8048,8372,96.13 %
179,Mathematics,Fibonacci Numbers,False,7786,9698,80.28 %


## Output to JSON

In [163]:
os.makedirs('data', exist_ok=True)
df.to_json('data/output.json')